In [1]:
from pathlib import Path
import polars as pl
from alpha_research.data import load_trades

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
DATA_ROOT = PROJECT_ROOT / "data"

DATA_ROOT

PosixPath('/Users/sumithsalluri/alpha_research/data')

In [17]:
spot = load_trades(
    market="spot",
    symbol="BTCUSDT",
    start_date="2024-01-01",
    end_date="2024-01-03",
    data_root=DATA_ROOT,
)

spot.head()

trade_id,price,quantity,quote_notional,time_ms,is_buyer_maker,is_best_match,exchange,market,symbol,source_date
i64,f64,f64,f64,i64,bool,bool,str,str,str,date
3344774379,42283.58,0.00069,29.17567,1704067200000,true,true,"""binance""","""spot""","""BTCUSDT""",2024-01-01
3344774380,42283.59,0.00144,60.88837,1704067200001,false,true,"""binance""","""spot""","""BTCUSDT""",2024-01-01
3344774381,42283.58,0.00102,43.129252,1704067200003,true,true,"""binance""","""spot""","""BTCUSDT""",2024-01-01
3344774382,42283.59,0.00219,92.601062,1704067200003,false,true,"""binance""","""spot""","""BTCUSDT""",2024-01-01
3344774383,42283.59,0.00163,68.922252,1704067200004,false,true,"""binance""","""spot""","""BTCUSDT""",2024-01-01


In [3]:
perp = load_trades(
    market="perp",
    symbol="BTCUSDT",
    start_date="2024-01-01",
    end_date="2024-01-03",
    data_root=DATA_ROOT,
)

perp.head()

trade_id,price,quantity,quote_notional,time_ms,is_buyer_maker,is_best_match,exchange,market,symbol,source_date
i64,f64,f64,f64,i64,bool,bool,str,str,str,date
4426785098,42314.0,0.033,1396.362,1704067200006,false,null,"""binance""","""perp""","""BTCUSDT""",2024-01-01
4426785099,42314.0,0.215,9097.51,1704067200006,false,null,"""binance""","""perp""","""BTCUSDT""",2024-01-01
4426785100,42314.0,0.1,4231.4,1704067200022,false,null,"""binance""","""perp""","""BTCUSDT""",2024-01-01
4426785101,42314.0,0.512,21664.768,1704067200022,false,null,"""binance""","""perp""","""BTCUSDT""",2024-01-01
4426785102,42314.0,0.007,296.198,1704067200022,false,null,"""binance""","""perp""","""BTCUSDT""",2024-01-01


In [4]:
def make_trade_bars(trades: pl.Dataframe, prefix: str) -> pl.Dataframe :
    return (
        trades
        .with_columns(
            pl.from_epoch(pl.col("time_ms"), time_unit="ms").alias("ts"),
            pl.when(pl.col("is_buyer_maker")).then(-pl.col("quote_notional")).otherwise(pl.col("quote_notional")).alias("signed_quote")
        )
        .with_columns(
            pl.col("ts").dt.truncate("1s").alias("timestamp")
        )
        .sort(["timestamp", "time_ms", "trade_id"])
        .group_by("timestamp")
        .agg(
            pl.col("price").last().alias(f"{prefix}_price"),
            pl.col("quote_notional").sum().alias(f"{prefix}_quote_volume_1s"),
            pl.col("signed_quote").sum().alias(f"{prefix}_signed_quote_1s"),
            pl.len().alias(f"{prefix}_trade_count_1s"),
        )
        .sort("timestamp")
    )

In [5]:
spot_1s = make_trade_bars(spot, "spot")
perp_1s = make_trade_bars(perp, "perp")

In [6]:
start = max(spot_1s["timestamp"].min(), perp_1s["timestamp"].min())
end = min(spot_1s["timestamp"].max(), perp_1s["timestamp"].max())

grid = pl.DataFrame(
  {
      "timestamp": pl.datetime_range(
          start,
          end,
          interval="1s",
          eager=True,
      )
  }
)

In [7]:
data = (
  grid
  .join(spot_1s, on="timestamp", how="left")
  .join(perp_1s, on="timestamp", how="left")
  .sort("timestamp")
  .with_columns(
      pl.col("spot_price").forward_fill(),
      pl.col("perp_price").forward_fill(),
      pl.col("spot_quote_volume_1s").fill_null(0),
      pl.col("spot_signed_quote_1s").fill_null(0),
      pl.col("spot_trade_count_1s").fill_null(0),
  )
  .with_columns(
      pl.when(pl.col("spot_quote_volume_1s") > 0)
      .then(pl.col("spot_signed_quote_1s") / pl.col("spot_quote_volume_1s"))
      .otherwise(0.0)
      .alias("spot_imbalance_1s")
  )
)

In [8]:
data

timestamp,spot_price,spot_quote_volume_1s,spot_signed_quote_1s,spot_trade_count_1s,perp_price,perp_quote_volume_1s,perp_signed_quote_1s,perp_trade_count_1s,spot_imbalance_1s
datetime[μs],f64,f64,f64,u32,f64,f64,f64,u32,f64
2024-01-01 00:00:00,42283.58,3224.123141,-1821.999296,21,42313.9,68125.5354,64232.6566,14,-0.565115
2024-01-01 00:00:01,42283.58,58151.769471,5851.209369,51,42313.9,null,null,null,0.10062
2024-01-01 00:00:02,42283.58,7977.643051,-7872.779748,9,42313.9,null,null,null,-0.986855
2024-01-01 00:00:03,42283.59,35301.720464,8618.244469,39,42317.9,811016.4709,-471725.0553,175,0.244131
2024-01-01 00:00:04,42283.59,10067.297927,-6978.058842,17,42310.3,458651.9848,-343642.744,106,-0.693141
…,…,…,…,…,…,…,…,…,…
2024-01-03 23:59:55,42845.22,8683.012311,-8465.358542,9,42849.7,102414.5491,-95472.6223,39,-0.974933
2024-01-03 23:59:56,42845.23,5506.46794,-3233.95694,18,42849.4,19710.7662,16454.2118,9,-0.587302
2024-01-03 23:59:57,42845.22,963.16055,-923.742939,5,42849.4,10283.8733,4542.0537,6,-0.959075


In [10]:
research = (
  data
  .with_columns(
      # spot features
      (pl.col("spot_price") / pl.col("spot_price").shift(1)).log().alias("spot_ret_1s"),
      (pl.col("spot_price") / pl.col("spot_price").shift(5)).log().alias("spot_ret_5s"),

      # perp same-period returns, useful for diagnostics
      (pl.col("perp_price") / pl.col("perp_price").shift(1)).log().alias("perp_ret_1s"),
      (pl.col("perp_price") / pl.col("perp_price").shift(5)).log().alias("perp_ret_5s"),

      # labels: future perp returns
      (pl.col("perp_price").shift(-1) / pl.col("perp_price")).log().alias("future_perp_ret_1s"),
      (pl.col("perp_price").shift(-5) / pl.col("perp_price")).log().alias("future_perp_ret_5s"),
      (pl.col("perp_price").shift(-30) / pl.col("perp_price")).log().alias("future_perp_ret_30s"),
  )
  .select(
      "timestamp",
      "spot_price",
      "perp_price",
      "spot_ret_1s",
      "spot_ret_5s",
      "spot_signed_quote_1s",
      "spot_quote_volume_1s",
      "spot_imbalance_1s",
      "spot_trade_count_1s",
      "perp_ret_1s",
      "perp_ret_5s",
      "future_perp_ret_1s",
      "future_perp_ret_5s",
      "future_perp_ret_30s",
  )
  .drop_nulls()
)

In [11]:
research.head()

timestamp,spot_price,perp_price,spot_ret_1s,spot_ret_5s,spot_signed_quote_1s,spot_quote_volume_1s,spot_imbalance_1s,spot_trade_count_1s,perp_ret_1s,perp_ret_5s,future_perp_ret_1s,future_perp_ret_5s,future_perp_ret_30s
datetime[μs],f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64
2024-01-01 00:00:05,42276.84,42303.1,-0.00016,-0.000159,-78755.200477,103655.429482,-0.759779,85,-0.00017,-0.000255,-0.000035,-0.000191,0.00047
2024-01-01 00:00:06,42273.21,42301.6,-0.000086,-0.000245,2327.366337,30055.943326,0.077434,50,-0.000035,-0.000291,-0.000154,-0.000154,0.000506
2024-01-01 00:00:07,42269.53,42295.1,-0.000087,-0.000332,-25351.103555,44437.490378,-0.570489,36,-0.000154,-0.000444,-0.000002,-0.00013,0.000648
2024-01-01 00:00:08,42269.52,42295.0,-2.3658e-7,-0.000333,-5806.98664,5952.393824,-0.975572,11,-0.000002,-0.000541,0.000002,-0.000125,0.000626
2024-01-01 00:00:09,42269.52,42295.1,0.0,-0.000333,-227.832626,963.322448,-0.236507,9,0.000002,-0.000359,-0.000002,0.000149,0.000667


In [12]:
research.select(
  pl.corr("spot_ret_1s", "future_perp_ret_1s").alias("spot_ret_1s_vs_future_perp_1s"),
  pl.corr("spot_ret_5s", "future_perp_ret_5s").alias("spot_ret_5s_vs_future_perp_5s"),
  pl.corr("spot_imbalance_1s", "future_perp_ret_5s").alias("spot_imbalance_vs_future_perp_5s"),
)

spot_ret_1s_vs_future_perp_1s,spot_ret_5s_vs_future_perp_5s,spot_imbalance_vs_future_perp_5s
f64,f64,f64
-0.000504,0.030189,0.035129


In [13]:
deciles = (
  research
  .with_columns(
      pl.col("spot_imbalance_1s")
      .qcut(10, labels=[str(i) for i in range(10)])
      .alias("imbalance_decile")
  )
  .group_by("imbalance_decile")
  .agg(
      pl.len().alias("rows"),
      pl.col("spot_imbalance_1s").mean().alias("avg_imbalance"),
      pl.col("future_perp_ret_1s").mean().alias("avg_future_perp_1s"),
      pl.col("future_perp_ret_5s").mean().alias("avg_future_perp_5s"),
      pl.col("future_perp_ret_30s").mean().alias("avg_future_perp_30s"),
  )
  .sort("imbalance_decile")
)

deciles

imbalance_decile,rows,avg_imbalance,avg_future_perp_1s,avg_future_perp_5s,avg_future_perp_30s
cat,u32,f64,f64,f64,f64
"""0""",30753,-1.0,-0.000004,-0.000007,-0.000006
"""1""",21080,-0.985147,-0.000011,-0.000021,-0.00002
"""2""",25917,-0.891028,-0.000008,-0.000016,-0.000022
"""3""",25916,-0.630281,-0.000003,-0.000005,-8.0068e-7
"""4""",31251,-0.170545,-0.000001,0.000002,2.7110e-7
"""5""",20582,0.188211,-8.6838e-7,4.4573e-7,-0.000005
"""6""",25916,0.579253,0.000005,0.000006,0.000008
"""7""",25917,0.857056,0.000008,0.000012,0.00002
"""8""",25916,0.97503,0.00001,0.000019,0.000019


In [15]:
ret_deciles = (
  research
  .with_columns(
      pl.col("spot_ret_5s")
      .qcut(10, labels=[str(i) for i in range(10)], allow_duplicates=True)
      .alias("spot_ret_5s_decile")
  )
  .group_by("spot_ret_5s_decile")
  .agg(
      pl.len().alias("rows"),
      pl.col("spot_ret_5s").mean().alias("avg_spot_ret_5s"),
      pl.col("future_perp_ret_1s").mean().alias("avg_future_perp_1s"),
      pl.col("future_perp_ret_5s").mean().alias("avg_future_perp_5s"),
      pl.col("future_perp_ret_30s").mean().alias("avg_future_perp_30s"),
  )
  .sort("spot_ret_5s_decile")
)

ret_deciles

spot_ret_5s_decile,rows,avg_spot_ret_5s,avg_future_perp_1s,avg_future_perp_5s,avg_future_perp_30s
cat,u32,f64,f64,f64,f64
"""0""",25917,-0.00041,-0.000007,-0.000021,-0.000042
"""1""",25916,-0.00013,-0.000005,-0.000011,-0.000006
"""2""",25917,-0.00004,-0.000003,-0.000008,-0.000002
"""3""",77744,-7.3778e-8,-0.000001,-0.000002,1.1523e-7
"""5""",5,2.1815e-7,-0.000012,-0.000041,-0.000538
"""6""",25916,2.5395e-7,0.000002,0.000003,0.000007
"""7""",25917,0.00004,0.000004,0.000009,0.000017
"""8""",25916,0.00013,0.000005,0.000014,0.000011
"""9""",25917,0.000412,0.000008,0.000023,0.00003


In [18]:
threshold = 0.95

strategy = (
  research
  .with_columns(
      pl.when(pl.col("spot_imbalance_1s") >= threshold)
      .then(1)
      .when(pl.col("spot_imbalance_1s") <= -threshold)
      .then(-1)
      .otherwise(0)
      .alias("position")
  )
  .with_columns(
      (pl.col("position") * pl.col("future_perp_ret_5s")).alias("gross_pnl_5s")
  )
)

In [19]:
strategy

timestamp,spot_price,perp_price,spot_ret_1s,spot_ret_5s,spot_signed_quote_1s,spot_quote_volume_1s,spot_imbalance_1s,spot_trade_count_1s,perp_ret_1s,perp_ret_5s,future_perp_ret_1s,future_perp_ret_5s,future_perp_ret_30s,position,gross_pnl_5s
datetime[μs],f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,i32,f64
2024-01-01 00:00:05,42276.84,42303.1,-0.00016,-0.000159,-78755.200477,103655.429482,-0.759779,85,-0.00017,-0.000255,-0.000035,-0.000191,0.00047,0,-0.0
2024-01-01 00:00:06,42273.21,42301.6,-0.000086,-0.000245,2327.366337,30055.943326,0.077434,50,-0.000035,-0.000291,-0.000154,-0.000154,0.000506,0,-0.0
2024-01-01 00:00:07,42269.53,42295.1,-0.000087,-0.000332,-25351.103555,44437.490378,-0.570489,36,-0.000154,-0.000444,-0.000002,-0.00013,0.000648,0,-0.0
2024-01-01 00:00:08,42269.52,42295.0,-2.3658e-7,-0.000333,-5806.98664,5952.393824,-0.975572,11,-0.000002,-0.000541,0.000002,-0.000125,0.000626,-1,0.000125
2024-01-01 00:00:09,42269.52,42295.1,0.0,-0.000333,-227.832626,963.322448,-0.236507,9,0.000002,-0.000359,-0.000002,0.000149,0.000667,0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-01-03 23:59:25,42829.87,42836.9,-2.3348e-7,0.0,-3236.65327,3285.479333,-0.985139,6,0.0,-0.000002,0.0,0.0,0.000299,-1,-0.0
2024-01-03 23:59:26,42829.88,42836.9,2.3348e-7,2.3348e-7,-1054.471165,3058.909549,-0.344721,5,0.0,0.0,0.0,0.000002,0.000292,0,0.0
2024-01-03 23:59:27,42829.88,42836.9,0.0,0.0,0.0,0.0,0.0,0,0.0,-0.000002,0.0,0.000002,0.000292,0,0.0


In [20]:
strategy.select(
  pl.len().alias("rows"),
  (pl.col("position") != 0).sum().alias("trades"),
  pl.col("gross_pnl_5s").mean().alias("avg_gross_pnl_per_row"),
  pl.col("gross_pnl_5s").sum().alias("total_gross_log_return"),
  pl.col("gross_pnl_5s").filter(pl.col("position") != 0).mean().alias("avg_gross_pnl_when_trading"),
)

rows,trades,avg_gross_pnl_per_row,total_gross_log_return,avg_gross_pnl_when_trading
u32,u32,f64,f64,f64
259165,103103,0.000005,1.397403,0.000014


In [21]:
strategy.group_by("position").agg(
  pl.len().alias("rows"),
  pl.col("future_perp_ret_5s").mean().alias("avg_future_perp_5s"),
  pl.col("gross_pnl_5s").mean().alias("avg_gross_pnl_5s"),
)

position,rows,avg_future_perp_5s,avg_gross_pnl_5s
i32,u32,f64,f64
0,156062,4.3499e-7,0.0
1,48451,0.000014,0.000014
-1,54652,-0.000013,0.000013


In [23]:
round_trip_cost = 0.0005  # 5 bps

strategy_costs = strategy.with_columns(
  pl.when(pl.col("position") != 0)
  .then(pl.col("gross_pnl_5s") - round_trip_cost)
  .otherwise(0)
  .alias("net_pnl_5s")
)

strategy_costs.select(
  pl.col("gross_pnl_5s").filter(pl.col("position") != 0).mean().alias("avg_gross_when_trading"),
  pl.col("net_pnl_5s").filter(pl.col("position") != 0).mean().alias("avg_net_when_trading"),
  pl.col("net_pnl_5s").sum().alias("total_net_log_return"),
)

avg_gross_when_trading,avg_net_when_trading,total_net_log_return
f64,f64,f64
0.000014,-0.000486,-50.154097
